# Progetto: Timed GNN per la risoluzione di puzzle scacchistici (Lichess)
### Fase 2 — Costruzione del grafo (PyTorch Geometric)

Questo notebook prende i dataset prodotti nella **Fase 1** (`puzzles_train.parquet`, ecc.) e li trasforma
in grafi `torch_geometric.data.Data`, pronti per l'addestramento di una GNN.

**Rappresentazione scelta:**
- **Nodi**: le 64 caselle della scacchiera. Feature: tipo di pezzo (6, one-hot) x colore (2) + flag
  "casella vuota" + coordinate normalizzate (file, rank).
- **Archi**: le mosse legali nella posizione (dopo la mossa di apertura dell'avversario), rappresentate
  come archi diretti `from_square -> to_square`. Aggiungiamo anche archi di "attacco" (relazioni pezzo
  attacca casella) come tipo di arco separato, per catturare tattica/minacce anche su archi che non sono
  mosse immediatamente legali (es. pezzo inchiodato).
- **Etichetta (target)**: indice, tra le mosse legali, della mossa di soluzione (il primo mossa del
  giocatore dopo la mossa di apertura dell'avversario). Impostiamo il problema come **classificazione
  multi-classe su un insieme di candidati variabile** (tipico nei modelli di move-prediction su scacchi).
- **Feature di timing**: il tempo di pensiero simulato (`simulated_think_time_sec`, Fase 1) viene
  aggiunto come **feature globale broadcast su tutti i nodi** (rappresenta la "pressione temporale"
  della posizione). Il parametro `include_timing` permette di disattivarla per l'ablation study
  richiesto dagli obiettivi del progetto (Fase 5, "Impact of timing information").

> **Nota sul formato Lichess**: la `FEN` è la posizione *prima* della mossa dell'avversario. La prima
> mossa in `Moves` è quindi la mossa dell'avversario che genera la posizione reale del puzzle; il grafo
> viene costruito **dopo** quella mossa, e la mossa target è la seconda della lista.


## 0. Setup

In [ ]:
# Installazione PyTorch Geometric (Colab). Il comando si adatta automaticamente
# alla versione di torch e CUDA installate sul runtime Colab.
import torch
print("Torch version:", torch.__version__, "| CUDA disponibile:", torch.cuda.is_available())

!pip install -q python-chess
!pip install -q torch_geometric


In [ ]:
import os
import numpy as np
import pandas as pd
import chess
import torch
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader

DATA_DIR = "/content/data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Caricamento dei dataset dalla Fase 1

Se questo notebook viene eseguito in una sessione Colab diversa da quella della Fase 1, i file Parquet
non saranno presenti: in tal caso ri-esegui prima il notebook `01_data_pipeline.ipynb`, oppure carica
manualmente i file in `/content/data/`.


In [ ]:
required_files = ["puzzles_train.parquet", "puzzles_val.parquet", "puzzles_test.parquet"]
missing = [f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    raise FileNotFoundError(
        f"File mancanti: {missing}. Esegui prima il notebook 01_data_pipeline.ipynb "
        f"oppure carica i file Parquet in {DATA_DIR}."
    )

df_train = pd.read_parquet(os.path.join(DATA_DIR, "puzzles_train.parquet"))
df_val = pd.read_parquet(os.path.join(DATA_DIR, "puzzles_val.parquet"))
df_test = pd.read_parquet(os.path.join(DATA_DIR, "puzzles_test.parquet"))

print(f"Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")
df_train.head(3)


## 2. Encoding della posizione: nodi e feature

Definiamo le costanti e le funzioni di encoding delle caselle.


In [ ]:
PIECE_TYPES = [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]
NUM_PIECE_FEATS = len(PIECE_TYPES) * 2 + 1  # 6 tipi x 2 colori + flag 'vuota'
NUM_POS_FEATS = 2                            # file, rank normalizzati
NUM_TIMING_FEATS = 1                         # tempo di pensiero simulato (broadcast)

def square_node_features(board, square):
    """Vettore di feature per una singola casella."""
    feat = [0.0] * NUM_PIECE_FEATS
    piece = board.piece_at(square)
    if piece is None:
        feat[-1] = 1.0
    else:
        idx = PIECE_TYPES.index(piece.piece_type)
        color_offset = 0 if piece.color == chess.WHITE else len(PIECE_TYPES)
        feat[idx + color_offset] = 1.0

    file_ = chess.square_file(square) / 7.0
    rank_ = chess.square_rank(square) / 7.0
    return feat + [file_, rank_]


## 3. Encoding degli archi: mosse legali + relazioni di attacco

Due tipi di arco, distinti tramite `edge_attr` (one-hot sul tipo):
- **tipo 0 — mossa legale**: `from_square -> to_square` per ogni mossa legale del giocatore di turno
- **tipo 1 — attacco**: per ogni pezzo, le caselle che attacca (`board.attacks(square)`), incluse quelle
  della propria squadra (utile per modellare difese/pin, coerente con la metodologia del progetto)

Le mosse legali sono anche i **candidati** tra cui il modello deve scegliere quella corretta (policy head).


In [ ]:
def build_edges(board):
    """Ritorna edge_index, edge_attr, e la lista delle mosse legali (allineata agli archi di tipo 'mossa legale')."""
    edge_src, edge_dst, edge_type = [], [], []

    legal_moves = list(board.legal_moves)
    move_edge_positions = []  # indice dell'arco corrispondente a ciascuna mossa legale, nell'ordine di legal_moves

    # Tipo 0: mosse legali
    for move in legal_moves:
        edge_src.append(move.from_square)
        edge_dst.append(move.to_square)
        edge_type.append(0)
        move_edge_positions.append(len(edge_src) - 1)

    # Tipo 1: relazioni di attacco (per tutti i pezzi presenti sulla scacchiera)
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is None:
            continue
        for target_sq in board.attacks(square):
            edge_src.append(square)
            edge_dst.append(target_sq)
            edge_type.append(1)

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_type_t = torch.tensor(edge_type, dtype=torch.long)
    edge_attr = torch.nn.functional.one_hot(edge_type_t, num_classes=2).float()

    return edge_index, edge_attr, legal_moves, move_edge_positions


## 4. Costruzione del target (etichetta)

Il target è l'indice, tra gli archi di tipo "mossa legale" (nell'ordine di `legal_moves`), corrispondente
alla mossa di soluzione. Questo indice verrà usato per la loss di classificazione (policy head) nella
fase di training: il modello produrrà uno score per ciascuna mossa legale (arco tipo 0) e useremo
cross-entropy sull'indice corretto.


In [ ]:
def get_position_and_target(fen, moves_str):
    """Applica la mossa di apertura dell'avversario (se presente) e individua la mossa target."""
    board = chess.Board(fen)
    moves = moves_str.split()

    if len(moves) >= 2:
        opening_move = chess.Move.from_uci(moves[0])
        board.push(opening_move)
        target_uci = moves[1]
    else:
        target_uci = moves[0]

    return board, chess.Move.from_uci(target_uci)


## 5. Dataset PyTorch Geometric

Costruiamo i grafi **on-the-fly** (non pre-salvati su disco) per due motivi pratici:
1. Il numero di puzzle è grande (centinaia di migliaia) e salvare tutti i grafi occuperebbe molto spazio
2. La costruzione è rapida (poche centinaia di microsecondi per posizione)

Il flag `include_timing` implementa l'ablation study richiesto dalla metodologia (modello con/senza
feature temporale).


In [ ]:
class ChessPuzzleGraphDataset(Dataset):
    def __init__(self, df, include_timing=True):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.include_timing = include_timing

    def len(self):
        return len(self.df)

    def get(self, idx):
        row = self.df.iloc[idx]
        board, target_move = get_position_and_target(row["FEN"], row["Moves"])

        # --- Feature dei nodi ---
        node_feats = [square_node_features(board, sq) for sq in chess.SQUARES]
        node_feats = torch.tensor(node_feats, dtype=torch.float)

        if self.include_timing:
            think_time = float(row["simulated_think_time_sec"])
            # normalizzazione semplice (min-max approssimato su range atteso 0-15s)
            think_time_norm = min(think_time, 15.0) / 15.0
            timing_col = torch.full((node_feats.size(0), 1), think_time_norm, dtype=torch.float)
        else:
            timing_col = torch.zeros((node_feats.size(0), 1), dtype=torch.float)

        node_feats = torch.cat([node_feats, timing_col], dim=1)

        # --- Archi ---
        edge_index, edge_attr, legal_moves, move_edge_positions = build_edges(board)

        # --- Target ---
        try:
            target_local_idx = legal_moves.index(target_move)
        except ValueError:
            # Non dovrebbe succedere se la Fase 1 ha validato correttamente i dati;
            # in caso di dati sporchi, segnaliamo con indice -1 (da filtrare a monte).
            target_local_idx = -1

        num_candidate_moves = len(legal_moves)

        data = Data(
            x=node_feats,
            edge_index=edge_index,
            edge_attr=edge_attr,
        )
        data.y = torch.tensor([target_local_idx], dtype=torch.long)
        data.num_candidate_moves = torch.tensor([num_candidate_moves], dtype=torch.long)
        data.mate_n = torch.tensor([int(row["mate_n"])], dtype=torch.long)
        data.puzzle_id = row["PuzzleId"]

        return data


## 6. Verifica su un piccolo campione

Costruiamo qualche grafo di esempio e controlliamo che tutto sia coerente (nessun target = -1, forme dei
tensori corrette) prima di lanciare la costruzione sull'intero dataset.


In [ ]:
sample_df = df_train.sample(200, random_state=0).reset_index(drop=True)
sample_ds = ChessPuzzleGraphDataset(sample_df, include_timing=True)

invalid_count = 0
for i in range(len(sample_ds)):
    d = sample_ds.get(i)
    if d.y.item() == -1:
        invalid_count += 1

print(f"Grafi con target non trovato: {invalid_count} / {len(sample_ds)}")

d0 = sample_ds.get(0)
print("\nEsempio di grafo:")
print(" - x (feature nodi):", d0.x.shape)
print(" - edge_index:", d0.edge_index.shape)
print(" - edge_attr:", d0.edge_attr.shape)
print(" - y (target):", d0.y, "| num mosse candidate:", d0.num_candidate_moves.item())
print(" - mate_n:", d0.mate_n.item())


In [ ]:
# Se invalid_count non e' zero, isoliamo e ispezioniamo i casi problematici
if invalid_count > 0:
    bad_rows = []
    for i in range(len(sample_ds)):
        if sample_ds.get(i).y.item() == -1:
            bad_rows.append(sample_df.iloc[i])
    bad_df = pd.DataFrame(bad_rows)
    print(bad_df[["PuzzleId", "FEN", "Moves"]])
else:
    print("Nessun caso problematico nel campione: la pipeline di encoding e' consistente.")


## 7. DataLoader

`torch_geometric.loader.DataLoader` gestisce automaticamente il batching di grafi con numero variabile
di nodi/archi (qui i nodi sono fissi a 64, ma il numero di archi/mosse candidate varia per posizione).

Nota: dato che il numero di mosse candidate (`num_candidate_moves`) varia da grafo a grafo, la policy
head nel modello (prossima fase) dovrà produrre uno score per **ciascun arco di tipo 'mossa legale'** nel
batch, non un vettore di dimensione fissa — è l'approccio standard per la move-prediction su grafi.


In [ ]:
train_ds = ChessPuzzleGraphDataset(df_train, include_timing=True)
val_ds = ChessPuzzleGraphDataset(df_val, include_timing=True)
test_ds = ChessPuzzleGraphDataset(df_test, include_timing=True)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

batch = next(iter(train_loader))
print(batch)
print("\nNumero di grafi nel batch:", batch.num_graphs)
print("x shape (nodi totali nel batch, feature):", batch.x.shape)
print("edge_index shape:", batch.edge_index.shape)


## 8. Due varianti per l'ablation study (timing on/off)

Prepariamo già ora i due DataLoader (con e senza timing) che verranno usati nella Fase 5
(Ablation Study) per confrontare le performance del modello con e senza la feature temporale.


In [ ]:
train_ds_no_timing = ChessPuzzleGraphDataset(df_train, include_timing=False)
train_loader_no_timing = DataLoader(train_ds_no_timing, batch_size=32, shuffle=True)

batch_no_timing = next(iter(train_loader_no_timing))
print("Con timing    -> ultima colonna feature nodo (dovrebbe variare):",
      batch.x[:5, -1].tolist())
print("Senza timing  -> ultima colonna feature nodo (dovrebbe essere 0):",
      batch_no_timing.x[:5, -1].tolist())


## Riepilogo Fase 2 e prossimi passi

**Cosa abbiamo costruito:**
- `ChessPuzzleGraphDataset`: converte ogni riga (FEN, Moves, timing) in un grafo `torch_geometric.data.Data`
  con 64 nodi (caselle), archi di tipo "mossa legale" (target candidates) e "attacco", feature di timing
  opzionale (broadcast sui nodi)
- `train_loader`, `val_loader`, `test_loader` pronti per l'addestramento
- Varianti con/senza timing pronte per l'ablation study (Fase 5 del progetto)

**Prossimi passi:**
1. **Modello GNN "timed"**: architettura message-passing (es. `GATConv`/`GINEConv` di PyTorch Geometric,
   che supportano `edge_attr`) con policy head che assegna uno score a ciascun arco "mossa legale" del
   grafo, e training loop con cross-entropy sul target
2. **Baseline LLM**: prompting per lo stesso set di puzzle held-out
3. **Valutazione comparativa** per valore di `n` e ablation timing on/off

Dimmi quando vuoi procedere con il modello GNN (punto 1) e continuo da lì.
